# Instacart Online Grocery Basket Analysis
**Dataset**: yasserh/instacart-online-grocery-basket-analysis-dataset

> **Note**: This dataset is identical to `instacart-market-basket`. This notebook confirms that and runs a focused affinity analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

BASE1 = 'shardul/instacart-market-basket'
BASE2 = 'shardul/instacart-grocery-basket'


## 1. Verify Datasets Are Identical

In [ ]:
csv1 = sorted([f for f in os.listdir(BASE1) if f.endswith('.csv')])
csv2 = sorted([f for f in os.listdir(BASE2) if f.endswith('.csv')])

print('Files match:', csv1 == csv2)
for f in csv1:
    s1 = os.path.getsize(os.path.join(BASE1, f))
    s2 = os.path.getsize(os.path.join(BASE2, f))
    print(f'  {f}: {"IDENTICAL" if s1 == s2 else "DIFFERENT"} ({s1:,} vs {s2:,} bytes)')


## 2. Product Affinity Deep-Dive

Since the main analysis is in the market-basket notebook, here we focus on **product co-purchase affinity** — critical for planogram adjacency decisions.

In [ ]:
# Load data
products = pd.read_csv(f'{BASE2}/products.csv')
aisles = pd.read_csv(f'{BASE2}/aisles.csv')
departments = pd.read_csv(f'{BASE2}/departments.csv')
op_prior = pd.read_csv(f'{BASE2}/order_products__prior.csv')
prod_full = products.merge(aisles, on='aisle_id').merge(departments, on='department_id')

print(f'Loaded {len(op_prior):,} order-product pairs')


In [ ]:
# Top product pairs (co-purchased in same order)
# Sample for memory efficiency
from collections import Counter
from itertools import combinations

sample_ids = op_prior['order_id'].drop_duplicates().sample(100000, random_state=42)
sample = op_prior[op_prior['order_id'].isin(sample_ids)]

pair_counts = Counter()
for oid, grp in sample.groupby('order_id')['product_id']:
    prods = grp.values
    if len(prods) <= 30:  # skip very large baskets
        for a, b in combinations(sorted(prods), 2):
            pair_counts[(a, b)] += 1

print(f'Unique product pairs found: {len(pair_counts):,}')


In [ ]:
# Top 20 co-purchased product pairs
top_pairs = pd.DataFrame(
    [(a, b, cnt) for (a, b), cnt in pair_counts.most_common(20)],
    columns=['product_a', 'product_b', 'co_purchase_count']
)
top_pairs = top_pairs.merge(prod_full[['product_id','product_name']], left_on='product_a', right_on='product_id')
top_pairs = top_pairs.rename(columns={'product_name': 'name_a'}).drop('product_id', axis=1)
top_pairs = top_pairs.merge(prod_full[['product_id','product_name']], left_on='product_b', right_on='product_id')
top_pairs = top_pairs.rename(columns={'product_name': 'name_b'}).drop('product_id', axis=1)

top_pairs['pair'] = top_pairs['name_a'] + ' + ' + top_pairs['name_b']

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(20), top_pairs['co_purchase_count'].values, color=sns.color_palette('viridis', 20))
ax.set_yticks(range(20))
ax.set_yticklabels(top_pairs['pair'].values)
ax.set_xlabel('Co-purchase Count (100k order sample)')
ax.set_title('Top 20 Most Frequently Co-Purchased Product Pairs')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Aisle-Level Affinity

In [ ]:
# Aisle co-occurrence
sample_aisle = sample.merge(prod_full[['product_id','aisle']], on='product_id')
aisle_pairs = Counter()
for oid, grp in sample_aisle.groupby('order_id')['aisle']:
    aisles_in_basket = grp.unique()
    for a, b in combinations(sorted(aisles_in_basket), 2):
        aisle_pairs[(a, b)] += 1

top_aisle_pairs = pd.DataFrame(
    [(a, b, cnt) for (a, b), cnt in sorted(aisle_pairs.items(), key=lambda x: -x[1])[:25]],
    columns=['aisle_a', 'aisle_b', 'count']
)
top_aisle_pairs['pair'] = top_aisle_pairs['aisle_a'] + ' + ' + top_aisle_pairs['aisle_b']

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(25), top_aisle_pairs['count'].values, color=sns.color_palette('rocket', 25))
ax.set_yticks(range(25))
ax.set_yticklabels(top_aisle_pairs['pair'].values)
ax.set_xlabel('Co-occurrence Count')
ax.set_title('Top 25 Aisle Pairs Co-occurring in Baskets')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 4. Takeaway for Planogram AI

- The co-purchase pairs above directly feed **Product Affinity Modeling (Feature 1.2)** in the MVP
- Aisle-level affinity informs **department/aisle adjacency** rules for shelf layout
- High co-purchase pairs should be placed **near each other** on shelves to maximize cross-sell
